**Record of a July 2026 Colab run.** This notebook keeps the original outputs, though its first cell was replaced to clone this repository, so those outputs came from the code of the time. To reproduce the result today, use [`docs/reproducing.md`](../docs/reproducing.md).

# Phase 5 — seeds 47 and 71 reproducibility runs

This notebook repeats the successful seed-23 experiment without changing its declared configuration: 60 optimizer steps, eight generations, per-device batch 8, generation batch 8, shaping removed after group 40, canonical stochastic evaluation over generation seeds 2000–2049, and separate greedy evaluation. Seed 47 and seed 71 run in independent cells and artifact directories. Run and report both regardless of outcome.

## 1. Fetch the repository

In [ ]:
# Fetch the public repository.
import os
import subprocess
from pathlib import Path

repository_path = Path('/content/browser-control')
if not repository_path.exists():
    subprocess.run(['git', 'clone', '--branch', 'lfm-browser-control',
                    'https://github.com/DerrickKirimi/browser-control.git', str(repository_path)], check=True)
os.chdir(repository_path)


## 2. Install without replacing Colab's PyTorch

In [4]:
from importlib.metadata import version

torch_before = version('torch')
%pip install -q -e ".[environment,model,training]" pytest
torch_after = version('torch')
print('PyTorch:', torch_before, '->', torch_after)
if torch_after != torch_before:
    raise RuntimeError('Project installation replaced Colab PyTorch')
%pip uninstall -y torchao
!python -m playwright install --with-deps chromium

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.6/191.6 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.8/37.8 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 137.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━

## 3. Validate GPU, decoder parity, and pinned TRL batching

In [5]:
!python scripts/check_colab_gpu.py
!python -m pytest -q tests/test_trl_contract.py tests/test_training.py tests/test_transformers_policy.py

{
  "cudnn_version": 91900,
  "device": {
    "compute_capability": "7.5",
    "index": 0,
    "name": "Tesla T4",
    "supports_bfloat16": false,
    "total_memory_bytes": 15637086208,
    "training_dtype": "float16"
  },
  "torch_cuda_version": "12.8",
  "torch_version": "2.11.0+cu128"
}
......................                                                   [100%]


## 4. Run seed 47

This cell is independent of seed 71. If Colab disconnects afterward, seed 47's artifacts remain identifiable.

In [6]:
!python scripts/train_controlled.py --seed 47 --max-steps 60 --curriculum-steps 40 --num-generations 8 --per-device-train-batch-size 8 --generation-batch-size 8 --output-dir artifacts/phase5/full-group-seed-47

2026-07-22 12:02:25.807462: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
tokenizer_config.json: 91.6kB [00:00, 176MB/s]
tokenizer.json: 4.73MB [00:00, 178MB/s]
special_tokens_map.json: 100% 434/434 [00:00<00:00, 4.17MB/s]
chat_template.jinja: 1.38kB [00:00, 9.34MB/s]
config.json: 100% 999/999 [00:00<00:00, 10.3MB/s]
model.safetensors: 100% 709M/709M [00:12<00:00, 57.5MB/s]
generation_config.json: 100% 137/137 [00:00<00:00, 1.30MB/s]
  0% 0/60 [00:00<?, ?it/s]Could not estimate the number of tokens of the input, floating-point operations will not be computed
{'loss': 0.0, 'grad_norm': 5.206045627593994, 'learning_rate': 0.0001, 'num_tokens': 1182.0, 'completions/mean_length': 10.75, 'completions/min_length': 5.0, 'completions/max_length': 15.0,

## 5. Run seed 71



In [7]:
!python scripts/train_controlled.py --seed 71 --max-steps 60 --curriculum-steps 40 --num-generations 8 --per-device-train-batch-size 8 --generation-batch-size 8 --output-dir artifacts/phase5/full-group-seed-71

2026-07-22 12:23:34.904001: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
  0% 0/60 [00:00<?, ?it/s]Could not estimate the number of tokens of the input, floating-point operations will not be computed
{'loss': -0.155, 'grad_norm': 3.862687587738037, 'learning_rate': 0.0001, 'num_tokens': 1265.0, 'completions/mean_length': 21.125, 'completions/min_length': 5.0, 'completions/max_length': 32.0, 'completions/clipped_ratio': 0.375, 'completions/mean_terminated_length': 14.600000381469727, 'completions/min_terminated_length': 5.0, 'completions/max_terminated_length': 29.0, 'rewards/click_test_reward/mean': 0.1635918766260147, 'rewards/click_test_reward/std': 0.33861804008483887, 'reward': 0.1635918766260147, 'reward_std': 0.3386180102825165, 'frac_re

## 6. Validate and download both reports and adapters

In [8]:
import json
import shutil
from pathlib import Path

from google.colab import files

expected_batching = {
    'generation_batch_size': 8,
    'gradient_accumulation_steps': 1,
    'num_generations': 8,
    'per_device_train_batch_size': 8,
    'steps_per_generation': 1,
}
for seed in (47, 71):
    output_dir = Path(f'artifacts/phase5/full-group-seed-{seed}')
    run_stem = f'phase5-click-test-seed-{seed:04d}-groups-0060-generations-0008'
    report_path = output_dir / f'{run_stem}-report.json'
    status_path = output_dir / 'run-status.json'
    if not status_path.is_file():
        raise RuntimeError(f'Seed {seed} has no status file')
    status = json.loads(status_path.read_text())
    if status.get('status') != 'success':
        if report_path.is_file():
            files.download(str(report_path))
        raise RuntimeError(f'Seed {seed} did not succeed: {status}')
    report = json.loads(report_path.read_text())
    if report['seed'] != seed or report['generation_batching'] != expected_batching:
        raise RuntimeError(f'Seed {seed} report does not match the frozen configuration')
    print(f'Seed {seed}:')
    print('  frozen stochastic:', report['frozen_evaluation']['stochastic']['successes'], '/ 50')
    print('  trained stochastic:', report['trained_evaluation']['stochastic']['successes'], '/ 50')
    print('  greedy success:', report['trained_evaluation']['greedy']['successes'], '/ 1')
    print('  success-rate delta:', report['strict_success_delta'])
    archive = shutil.make_archive(f'{run_stem}-adapter', 'zip', output_dir / 'adapter')
    files.download(str(report_path))
    files.download(archive)

Seed 47:
  frozen stochastic: 2 / 50
  trained stochastic: 50 / 50
  greedy success: 1 / 1
  success-rate delta: 0.96


[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

Seed 71:
  frozen stochastic: 2 / 50
  trained stochastic: 50 / 50
  greedy success: 1 / 1
  success-rate delta: 0.96


[browser download widget removed]

[browser download widget removed]

[browser download widget removed]

[browser download widget removed]